# CIFAR-10 CNN comparison

Three architectural recipes refactored from the [original Colab notebook](https://colab.research.google.com/drive/1UwTCG81bCByh1s3R_jvjYYqM9DQRqgvZ). Unlike the original, this notebook selects by a stratified validation split and **reserves the official test set until final selection**.

**Current status:** This corrected notebook has not yet been executed. Historical Colab validation scores are not unbiased test scores.

## Experimental protocol

CIFAR-10 official 50,000 training images → stratified 45,000 training / 5,000 validation. The official 10,000 test images are used **once**, only after the best validation-loss model is selected. Training recipes differ in depth, activations, optimizer and batch size; this is not a controlled batch-normalization ablation.

In [ ]:
from cifar10_experiment import MODEL_CONFIGS, run_experiments

MODEL_CONFIGS

## Run experiments

Start with a smoke test (one epoch on small subsets; **no test evaluation**). Set `SMOKE = False` after installing dependencies to run the full 50/100/150-epoch comparison. This can consume substantial CPU/GPU time.

In [ ]:
SMOKE = True
output_directory = run_experiments(smoke=SMOKE)
print(output_directory)

## Inspect validation-based model selection

The comparison contains checkpoint values from the **best validation-loss epoch** for each model. These are not official test metrics.

In [ ]:
import csv

with (output_directory / 'validation_comparison.csv').open(encoding='utf-8', newline='') as file:
    for item in csv.DictReader(file):
        print(item)

## Final test results

`test_metrics.json` is generated only for a completed full run. The test set is deliberately unavailable in smoke mode.

In [ ]:
import json

test_file = output_directory / 'test_metrics.json'
if test_file.exists() and not SMOKE:
    with test_file.open(encoding='utf-8') as file:
        result = json.load(file)
    print({name: result[name] for name in ('selected_model', 'test_accuracy', 'macro_f1')})
else:
    print('No final held-out test evaluation is available for this run.')